# Experiment: HuBERT-ECG Fine-tuning with Selective Layer Unfreezing

This notebook fine-tunes HuBERT-ECG (pretrained on 9.1M ECGs) by selectively unfreezing the last N transformer blocks. We compare unfreezing 4 vs 8 blocks as a parameter-efficiency ablation.

**Note — PatchTST removed:** PatchTST was initially evaluated but removed from the experimental pipeline for three reasons: (1) no pretrained ECG weights exist — it trains from random initialization only; (2) HuBERT-ECG is a stronger and better-motivated baseline as a domain-pretrained model; (3) the lead-wise Transformer is the novel architecture in this project and replaces PatchTST as the second comparison point.

In [1]:
import warnings
warnings.filterwarnings(
    "ignore",
    message=".*Torch was not compiled with flash attention.*"
)

In [2]:
from src.models.hubert_ecg_finetune import HuBERTECGClassifier
import torch

model = HuBERTECGClassifier(size="base", blocks_to_unfreeze=4)

dummy = torch.randn(4, 12, 1000)
with torch.no_grad():
    out = model(dummy)

print(f"Output shape: {out.shape}")  # must be (4, 5)
print("Forward pass OK")


KeyboardInterrupt



In [ ]:
# Setup
import sys, os, warnings
sys.path.append('../')
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
warnings.filterwarnings("ignore", category=FutureWarning)

import torch, pandas as pd, matplotlib.pyplot as plt

from src.preprocessing.label_utils  import load_all_labels
from src.preprocessing.dataset_full import ECGDatasetFull
from src.models.hubert_ecg_finetune import HuBERTECGClassifier
from src.training.train_peft        import run_experiment
from src.utils.config               import CFG

DATA_PATH = CFG['data']['path']
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

In [3]:
# Load labels
Y = load_all_labels(
    DATA_PATH + 'ptbxl_database.csv',
    DATA_PATH + 'scp_statements.csv'
)
train_df = Y[Y.strat_fold <  9]
val_df   = Y[Y.strat_fold == 9]
test_df  = Y[Y.strat_fold == 10]


Records with valid labels: 21388
Class distribution:
  NORM: 9514 (44.5%)
  MI: 5469 (25.6%)
  STTC: 5235 (24.5%)
  CD: 4898 (22.9%)
  HYP: 2649 (12.4%)


In [ ]:
# Build datasets — full 10-second records for HuBERT-ECG
train_ds_f = ECGDatasetFull(train_df, DATA_PATH)
val_ds_f   = ECGDatasetFull(val_df,   DATA_PATH)

print(f"Full train: {len(train_ds_f)} samples")
print(f"Full val:   {len(val_ds_f)} samples")

In [6]:
#  Experiment A: HuBERT-ECG 4 blocks
model_A = HuBERTECGClassifier(size=CFG['model']['hubert_size'], blocks_to_unfreeze=4)
auc_A, hist_A = run_experiment(
    model_A, train_ds_f, val_ds_f,
    experiment_name="hubert_ecg_blocks4",
    epochs=CFG['training']['epochs'], lr=CFG['training']['lr_pretrained'],  # lower LR for pretrained model
    batch_size=CFG['training']['batch_size_full'],                           # smaller batch — full records are larger
    save_dir=CFG['paths']['results']
)
del model_A; torch.cuda.empty_cache()


Loading HuBERT-ECG-base...
Loaded.
Unfrozen 64 tensors from last 4 blocks (layers 8–11)
Trainable: 28,551,173 / 93,323,397 (30.6%)

 Experiment : hubert_ecg_blocks4
 Device     : cuda
 Trainable  : 28,551,173  (30.6%)

Epoch 01/3  lr=5.00e-05
  train_loss=0.8513  val_loss=0.8420
  AUC (macro): 0.5163
  F1  (macro): 0.2704
  Per-class AUC:
    NORM : 0.406  ########
    MI   : 0.556  ###########
    STTC : 0.424  ########
    CD   : 0.638  ############
    HYP  : 0.558  ###########
Saved -> D:\GitHub\biosignal-xai\results/hubert_ecg_blocks4\best_adapter/checkpoint.pt
  * Best saved — AUC 0.5163

Epoch 02/3  lr=1.00e-04
  train_loss=0.5989  val_loss=0.5469
  AUC (macro): 0.8438
  F1  (macro): 0.6196
  Per-class AUC:
    NORM : 0.905  ##################
    MI   : 0.842  ################
    STTC : 0.899  #################
    CD   : 0.837  ################
    HYP  : 0.736  ##############
Saved -> D:\GitHub\biosignal-xai\results/hubert_ecg_blocks4\best_adapter/checkpoint.pt
  * Best save

In [7]:
#  Experiment B: HuBERT-ECG 8 blocks
model_B = HuBERTECGClassifier(size=CFG['model']['hubert_size'], blocks_to_unfreeze=8)
auc_B, hist_B = run_experiment(
    model_B, train_ds_f, val_ds_f,
    experiment_name="hubert_ecg_blocks8",
    epochs=CFG['training']['epochs'], lr=CFG['training']['lr_pretrained'],
    batch_size=CFG['training']['batch_size_full'],
    save_dir=CFG['paths']['results']
)
del model_B; torch.cuda.empty_cache()


Loading HuBERT-ECG-base...
Loaded.
Unfrozen 128 tensors from last 8 blocks (layers 4–11)
Trainable: 56,902,661 / 93,323,397 (61.0%)

 Experiment : hubert_ecg_blocks8
 Device     : cuda
 Trainable  : 56,902,661  (61.0%)

Epoch 01/3  lr=5.00e-05
  train_loss=0.8082  val_loss=0.8053
  AUC (macro): 0.5199
  F1  (macro): 0.2423
  Per-class AUC:
    NORM : 0.576  ###########
    MI   : 0.518  ##########
    STTC : 0.434  ########
    CD   : 0.487  #########
    HYP  : 0.584  ###########
Saved -> D:\GitHub\biosignal-xai\results/hubert_ecg_blocks8\best_adapter/checkpoint.pt
  * Best saved — AUC 0.5199

Epoch 02/3  lr=1.00e-04
  train_loss=0.5735  val_loss=0.6355
  AUC (macro): 0.8325
  F1  (macro): 0.5702
  Per-class AUC:
    NORM : 0.859  #################
    MI   : 0.856  #################
    STTC : 0.868  #################
    CD   : 0.843  ################
    HYP  : 0.737  ##############
Saved -> D:\GitHub\biosignal-xai\results/hubert_ecg_blocks8\best_adapter/checkpoint.pt
  * Best save

In [ ]:
#  Experiment C: HuBERT-ECG + LoRA r=8
from src.models.hubert_ecg_finetune import HuBERTECGPEFT

# Shape verification before training
_dummy = torch.randn(4, 12, 1000)
_m = HuBERTECGPEFT(rank=8, use_dora=False)
with torch.no_grad():
    _out = _m(_dummy)
assert _out.shape == (4, 5), f"Expected (4, 5), got {_out.shape}"
print(f"HuBERT PEFT output: {_out.shape} -- OK")
_m.backbone.print_trainable_parameters()
del _m, _dummy, _out

model_C = HuBERTECGPEFT(rank=8, use_dora=False)
auc_C, hist_C = run_experiment(
    model_C, train_ds_f, val_ds_f,
    experiment_name="hubert_ecg_lora_r8",
    epochs=CFG['training']['epochs'],
    lr=CFG['training']['lr_pretrained'],
    batch_size=CFG['training']['batch_size_full'],
    save_dir=CFG['paths']['results']
)
del model_C; torch.cuda.empty_cache()

In [ ]:
#  Experiment D: HuBERT-ECG + DoRA r=8
model_D = HuBERTECGPEFT(rank=8, use_dora=True)
auc_D, hist_D = run_experiment(
    model_D, train_ds_f, val_ds_f,
    experiment_name="hubert_ecg_dora_r8",
    epochs=CFG['training']['epochs'],
    lr=CFG['training']['lr_pretrained'],
    batch_size=CFG['training']['batch_size_full'],
    save_dir=CFG['paths']['results']
)
del model_D; torch.cuda.empty_cache()

In [ ]:
import json, os

dummy_path = os.path.join(CFG['paths']['results'], 'dummy_metrics.json')
with open(dummy_path) as f:
    dummy_metrics = json.load(f)

def best_auc(h): return max(e["auc_macro"] for e in h)
def best_f1(h):  return max(e["f1_macro"]  for e in h)

rows = [
    {"Model": "Dummy (prior)",    "Strategy": "Prior freq.",       "AUC": dummy_metrics['auc_macro'], "F1": dummy_metrics['f1_macro']},
    {"Model": "CNN (baseline)",   "Strategy": "Full training",     "AUC": 0.9068,             "F1": 0.6817},
    {"Model": "HuBERT-ECG-base", "Strategy": "Unfreeze 4 blocks", "AUC": best_auc(hist_A),   "F1": best_f1(hist_A)},
    {"Model": "HuBERT-ECG-base", "Strategy": "Unfreeze 8 blocks", "AUC": best_auc(hist_B),   "F1": best_f1(hist_B)},
    {"Model": "HuBERT-ECG-base", "Strategy": "LoRA r=8",          "AUC": best_auc(hist_C),   "F1": best_f1(hist_C)},
    {"Model": "HuBERT-ECG-base", "Strategy": "DoRA r=8",          "AUC": best_auc(hist_D),   "F1": best_f1(hist_D)},
]
results_df = pd.DataFrame(rows)
results_df["vs CNN"] = (results_df["AUC"] - 0.9068).map(
    lambda x: f"+{x:.4f}" if x > 0 else f"{x:.4f}"
)
print(results_df.to_string(index=False))

In [ ]:
# Parameter efficiency comparison table (key result for the paper)
TOTAL = 93_323_397   # confirmed HuBERT-ECG-base total params

print(f"{'Method':<28s}  {'Trainable':>14s}  {'% of total':>10s}")
print("-" * 58)
print(f"{'Full fine-tune':<28s}  {TOTAL:>14,}  {100.0:>9.1f}%")

m4 = HuBERTECGClassifier(size='base', blocks_to_unfreeze=4)
p4 = m4.count_parameters()
print(f"{'Selective (4 blocks)':<28s}  {p4['trainable']:>14,}  {100*p4['trainable']/TOTAL:>9.1f}%")
del m4

m8 = HuBERTECGClassifier(size='base', blocks_to_unfreeze=8)
p8 = m8.count_parameters()
print(f"{'Selective (8 blocks)':<28s}  {p8['trainable']:>14,}  {100*p8['trainable']/TOTAL:>9.1f}%")
del m8

ml = HuBERTECGPEFT(rank=8, use_dora=False)
pl = ml.count_parameters()
print(f"{'LoRA r=8':<28s}  {pl['trainable']:>14,}  {100*pl['trainable']/pl['total']:>9.1f}%")
del ml; torch.cuda.empty_cache()

md = HuBERTECGPEFT(rank=8, use_dora=True)
pd_ = md.count_parameters()
print(f"{'DoRA r=8':<28s}  {pd_['trainable']:>14,}  {100*pd_['trainable']/pd_['total']:>9.1f}%")
del md; torch.cuda.empty_cache()

In [ ]:
# Learning curves
fig, ax = plt.subplots(figsize=(12, 5))

experiments = [
    ("HuBERT 4 blocks", hist_A, "tab:blue"),
    ("HuBERT 8 blocks", hist_B, "tab:orange"),
    ("HuBERT LoRA r=8", hist_C, "tab:green"),
    ("HuBERT DoRA r=8", hist_D, "tab:red"),
]
for name, hist, color in experiments:
    ax.plot([e["epoch"] for e in hist],
            [e["auc_macro"] for e in hist], label=name, linewidth=2, color=color)

ax.axhline(y=0.9068, color='red',  linestyle='--', linewidth=1.5, label='CNN baseline')
ax.axhline(y=dummy_metrics['auc_macro'], color='gray', linestyle=':', linewidth=1.5, label='Dummy baseline')
ax.set_title("Validation AUC: HuBERT-ECG fine-tuning strategies")
ax.set_xlabel("Epoch"); ax.set_ylabel("AUC (macro)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CFG['paths']['figures'] + 'hubert_comparison.png', dpi=150)
plt.show()